The problem is that it is not easy to know how political parties may perfom in the 2026 local government elections. This project seeks to develop a Python-based machine learning solution that uses reliable historical South African local election data and other relevant supporting information to estimate the likely 2026 performance of political parties in one selected metropolitan municipality. The system will clean  and align historical data, identify important features, train and validate suitable forecasting models, estimate party vote totals and voter turnout and provide ward level estimation for 3 geographical comparable wards.

In [56]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
df=pd.read_csv("/content/ETH_Ward 2021.csv")
print(df.shape)

(32990, 11)


df head prints out the first 5 rows of the data

In [57]:
df.head()

,Province,Municipality,Ward,VotingDistrict,VotingStationName,RegisteredVoters,BallotType,SpoiltVotes,PartyName,TotalValidVotes,DateGenerated
0,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,2365,Ward,8,ABANTU BATHO CONGRESS,19,11/23/2021 4:11:07 PM
1,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,2365,Ward,8,ACADEMIC CONGRESS UNION,0,11/23/2021 4:11:07 PM
2,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,2365,Ward,8,ACTIONSA,0,11/23/2021 4:11:07 PM
3,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,2365,Ward,8,ACTIVISTS MOVEMENT OF SOUTH AFRICA,0,11/23/2021 4:11:07 PM
4,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,2365,Ward,8,ADVANCED DYNAMIC ALLIANCE,0,11/23/2021 4:11:07 PM


df.info() shows us the data structure

Observation: We can see that we have 32990 records

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32990 entries, 0 to 32989
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Province           32990 non-null  object
 1   Municipality       32990 non-null  object
 2   Ward               32990 non-null  object
 3   VotingDistrict     32990 non-null  int64 
 4   VotingStationName  32990 non-null  object
 5   RegisteredVoters   32990 non-null  int64 
 6   BallotType         32990 non-null  object
 7   SpoiltVotes        32990 non-null  int64 
 8   PartyName          32990 non-null  object
 9   TotalValidVotes    32990 non-null  int64 
 10  DateGenerated      32990 non-null  object
dtypes: int64(4), object(7)
memory usage: 2.8+ MB


In [59]:
df["BallotType"].value_counts()

,count
BallotType,
Ward,32990


In [60]:
df["PartyName"].value_counts()

,count
PartyName,
ACADEMIC CONGRESS UNION,875
ACTIVISTS MOVEMENT OF SOUTH AFRICA,875
AFRICAN MANTUNGWA COMMUNITY,875
AFRICAN CHRISTIAN DEMOCRATIC PARTY,875
AFRICA RESTORATION ALLIANCE,875
DEMOCRATIC ALLIANCE,875
DEMOCRATIC LIBERAL CONGRESS,875
AL JAMA-AH,875
AFRICAN NATIONAL CONGRESS,875


In [61]:
df["PartyName"].nunique()

53

In [62]:
df.nunique()

,0
Province,1
Municipality,1
Ward,111
VotingDistrict,875
VotingStationName,874
RegisteredVoters,781
BallotType,1
SpoiltVotes,77
PartyName,53
TotalValidVotes,769


df.describe calculates and gives us the summary of the numerical values

In [63]:
df.describe()

,VotingDistrict,RegisteredVoters,SpoiltVotes,TotalValidVotes
count,3.299000e+04,32990.000000,32990.000000,32990.000000
mean,4.342043e+07,2185.880115,16.501576,23.522734
std,1.021454e+05,1189.342444,19.547402,111.847546
min,4.335002e+07,107.000000,0.000000,0.000000
25%,4.336170e+07,1363.000000,4.000000,0.000000
50%,4.337166e+07,2057.000000,12.000000,0.000000
75%,4.340056e+07,2823.000000,22.000000,3.000000
max,4.407007e+07,7910.000000,221.000000,3163.000000


df.columns() helps to see all the columns in my data frame

Observation: My data frame contains 11 columns

In [64]:
df.columns

Index(['Province', 'Municipality', 'Ward', 'VotingDistrict',
       'VotingStationName', 'RegisteredVoters', 'BallotType', 'SpoiltVotes',
       'PartyName', 'TotalValidVotes', 'DateGenerated'],
      dtype='object')

df.isnull().sum checks for nun-null values

Observation: I can see that there are no null values

In [65]:
df.isnull().sum()

,0
Province,0
Municipality,0
Ward,0
VotingDistrict,0
VotingStationName,0
RegisteredVoters,0
BallotType,0
SpoiltVotes,0
PartyName,0
TotalValidVotes,0


print (df.duplicated().sum()) Checks for duplicates

Observation: I can clearly see that there are no duplicates

In [66]:
print(df.duplicated().sum())

0


df["Location"].value_counts() Checks how many times a value appears in a column

Observation: I can see that

In [67]:
df["Province"].value_counts()

,count
Province,
KwaZulu-Natal,32990


In [68]:
print(df["VotingDistrict"].nunique())

875


In [69]:
print(df["VotingDistrict"].value_counts().head())

VotingDistrict
43362042    43
43360512    43
43360578    43
43360523    43
43360589    43
Name: count, dtype: int64


In [70]:
df.groupby("VotingDistrict")["RegisteredVoters"].unique().value_counts()

,count
RegisteredVoters,
[1890],3
[913],3
[1748],3
[1908],3
[1600],3
...,...
[1751],1
[1060],1
[554],1


In [71]:
df.groupby("VotingDistrict")["SpoiltVotes"].unique().value_counts()

,count
SpoiltVotes,
[0],129
[7],37
[9],35
[12],33
[5],33
...,...
[190],1
[79],1
[77],1


In [72]:
model_df = df.copy()

In [73]:
model_df.nunique()

,0
Province,1
Municipality,1
Ward,111
VotingDistrict,875
VotingStationName,874
RegisteredVoters,781
BallotType,1
SpoiltVotes,77
PartyName,53
TotalValidVotes,769


In [74]:
model_df.head()

,Province,Municipality,Ward,VotingDistrict,VotingStationName,RegisteredVoters,BallotType,SpoiltVotes,PartyName,TotalValidVotes,DateGenerated
0,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,2365,Ward,8,ABANTU BATHO CONGRESS,19,11/23/2021 4:11:07 PM
1,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,2365,Ward,8,ACADEMIC CONGRESS UNION,0,11/23/2021 4:11:07 PM
2,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,2365,Ward,8,ACTIONSA,0,11/23/2021 4:11:07 PM
3,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,2365,Ward,8,ACTIVISTS MOVEMENT OF SOUTH AFRICA,0,11/23/2021 4:11:07 PM
4,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,2365,Ward,8,ADVANCED DYNAMIC ALLIANCE,0,11/23/2021 4:11:07 PM


In [75]:
import pandas as pd

# Corrected the typo "pronvice" -> "Province"
group_cols = [
    "Province",
    "Municipality",
    "Ward",
    "VotingDistrict",
    "VotingStationName"
]

# Fixed the indentation, removed stray parenthesis, and corrected column names
df["RegisteredVoters"] = pd.to_numeric(df["RegisteredVoters"], errors="coerce")
df["SpoiltVotes"] = pd.to_numeric(df["SpoiltVotes"], errors="coerce")
df["TotalValidVotes"] = pd.to_numeric(df["TotalValidVotes"], errors="coerce")

In [76]:
district_votes = (
    df.groupby(group_cols)["TotalValidVotes"].sum().reset_index()
)


In [77]:
district_votes = district_votes.rename(
    columns={"TotalValidVotes": "DistrictValidVotes"}
)


In [78]:
df = df.merge(
    district_votes, on=group_cols, how="left"
)

In [79]:
df["VoteShare"]=(
    df["TotalValidVotes"] / df["DistrictValidVotes"]
) * 100

In [80]:
party_votes = df.pivot_table(
    index=group_cols, columns="PartyName",
    values= "VoteShare",
    aggfunc="sum",
    fill_value=0
).reset_index()


In [81]:
district_info = (
    df.groupby(group_cols).agg(
        RegisteredVoters=("RegisteredVoters", "first"),
        SpoiltVotes=("SpoiltVotes", "first"),
        DistrictValidVotes=("DistrictValidVotes", "first")
    )
       .reset_index()
)


In [82]:
# Recreate model_df from district_info which contains all required district-level columns
model_df = district_info.copy()

# Corrected the assignment syntax to assign to the "Turnout" column
model_df["Turnout"] = (
    (
        model_df["DistrictValidVotes"]
      + model_df["SpoiltVotes"]
    )
     / model_df["RegisteredVoters"]
) * 100

In [83]:

model_df = district_info.merge(
    party_votes,
    on=group_cols,
    how="left"
)

model_df["Turnout"] = (
    (model_df["DistrictValidVotes"] + model_df["SpoiltVotes"])
    / model_df["RegisteredVoters"]
) * 100


print("ANC column exists:", "AFRICAN NATIONAL CONGRESS" in model_df.columns)


print(model_df["AFRICAN NATIONAL CONGRESS"].head())

ANC column exists: True
0    88.659794
1    96.049896
2    31.104651
3    85.612245
4    94.312796
Name: AFRICAN NATIONAL CONGRESS, dtype: float64


In [84]:
print(model_df["Turnout"].head())

0    41.353066
1    61.695347
2    39.722864
3    52.962963
4    66.094421
Name: Turnout, dtype: float64


In [85]:
print(model_df["Turnout"].describe())

count    875.000000
mean      43.629475
std       10.301845
min        9.573672
25%       36.988414
50%       43.269231
75%       49.888406
max      101.587302
Name: Turnout, dtype: float64


In [86]:
features = [
"AFRICAN NATIONAL CONGRESS",
"ECONOMIC FREEDOM FIGHTERS",
"INKATHA FREEDOM PARTY"
]


In [87]:
target = "Turnout"

In [88]:
features = [
    "AFRICAN NATIONAL CONGRESS",
    "ECONOMIC FREEDOM FIGHTERS",
    "INKATHA FREEDOM PARTY"
]

X = model_df[features]

In [89]:
print(X.head())

   AFRICAN NATIONAL CONGRESS  ECONOMIC FREEDOM FIGHTERS  INKATHA FREEDOM PARTY
0                  88.659794                   7.731959               0.206186
1                  96.049896                   2.598753               0.415800
2                  31.104651                   6.104651               3.779070
3                  85.612245                   5.306122               0.816327
4                  94.312796                   2.369668               0.189573


In [90]:
y= model_df["Turnout"]

In [91]:
print(y.head())

0    41.353066
1    61.695347
2    39.722864
3    52.962963
4    66.094421
Name: Turnout, dtype: float64


I am now testing my model using the train split method which uses
80% - of data for training
20% - of data for testing

In [92]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



Checking the sizes

In [93]:
print("Training X:", X_train.shape)
print("Testing X:", X_test.shape)
print("Training y:", y_train.shape)
print("Testing y:", y_test.shape)

Training X: (700, 3)
Testing X: (175, 3)
Training y: (700,)
Testing y: (175,)


In [94]:
random_state=42


In [95]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(y_pred[10])

39.24137629547957


Calculating the errors

In [96]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
rmse =np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print("MAE:", mae)
print("R2:", r2)

MAE: 6.615897214592883
R2: 0.2323806622408855


In [97]:
rsme = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE:", rsme)

RMSE: 9.364322322176177


In [98]:
print("ANC coefficient:", model.coef_[0])
print("EFF coefficient:", model.coef_[1])
print("IFP coefficient:", model.coef_[2])

ANC coefficient: -0.005667371566171315
EFF coefficient: -0.42968724164145694
IFP coefficient: -0.2520281639967805


In [99]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

accuracy = 100 - (mae / y_test.mean() * 100)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)
print("Approximate Accuracy:", accuracy, "%")

MAE: 6.615897214592883
RMSE: 9.364322322176177
R²: 0.2323806622408855
Approximate Accuracy: 84.945928520531 %


Forecasting

In [100]:

future_data = pd.DataFrame({
    "AFRICAN NATIONAL CONGRESS": [80],
    "ECONOMIC FREEDOM FIGHTERS": [10],
    "INKATHA FREEDOM PARTY": [5]
})
forecast = model.predict(future_data)

print("Forecasted voter turnout:", forecast[0], "%")

Forecasted voter turnout: 44.14427870061523 %


https://chatgpt.com/share/6ac3a4b9-30e0-83ea-833d-210e7de6acd1
chatgpt